# Integrating FAISS Retrievers into LangGraph

**FAISS** gives you fast vector search over chunked documents. **LangGraph** gives you an explicit graph for when retrieval runs — before generation, as a tool call, or in a corrective loop with re-querying.

Plain RAG is one function: `retrieve → prompt → answer`. Agent workflows need **control flow**:

```
Pattern A — Fixed pipeline          Pattern B — Retrieval tool
START → retrieve → generate → END     START → agent ⇄ ToolNode(FAISS) → END

Pattern C — Corrective RAG
START → retrieve → grade → rewrite? → retrieve → generate → END
```

This notebook builds **ShopCo Policy Support** with:

1. A **FAISS retriever** (TF-IDF embeddings, offline)
2. Three **LangGraph** patterns integrating that retriever
3. **State** that carries messages, hits, context, and grades
4. **Citations** and abstention on low scores
5. Optional **live LLM** generation

Self-contained — no references to other notebooks. Uses `langgraph`, `faiss`, and `scikit-learn`.

In [ ]:
"""
ShopCo Policy RAG — FAISS + LangGraph integration lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Annotated, Any, Literal

import faiss
import numpy as np
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from sklearn.feature_extraction.text import TfidfVectorizer
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


POLICY_DOCS: list[dict[str, str]] = [
    {
        "id": "pol-returns-01",
        "title": "Returns Policy",
        "text": (
            "Customers may return items within 30 days of delivery. Items must be unused "
            "and in original packaging. Refunds are processed within 5-7 business days "
            "after warehouse receipt. Final-sale items cannot be returned."
        ),
    },
    {
        "id": "pol-shipping-02",
        "title": "Shipping Policy",
        "text": (
            "Standard shipping is free on orders over $50. Express shipping is available "
            "at checkout. Tracking numbers are emailed when the carrier scans the package."
        ),
    },
    {
        "id": "pol-billing-03",
        "title": "Billing Policy",
        "text": (
            "Charges appear as SHOPCO* on credit card statements. Partial refunds may take "
            "up to two billing cycles to display. Contact support with order ID for disputes."
        ),
    },
    {
        "id": "faq-tracking-04",
        "title": "Tracking FAQ",
        "text": (
            "If order status is shipped, use the tracking number from your confirmation email. "
            "Processing orders do not yet have tracking numbers assigned."
        ),
    },
    {
        "id": "faq-warranty-05",
        "title": "Warranty FAQ",
        "text": (
            "Electronics include a 1-year manufacturer warranty. Warranty claims require "
            "proof of purchase and serial number. Warranty does not cover accidental damage."
        ),
    },
]

ORDERS_DB: dict[str, dict[str, Any]] = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "tracking": "1Z999AA10123456784"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "tracking": None},
}

print(f"Corpus: {len(POLICY_DOCS)} documents")

---

## 1. Why LangGraph for FAISS Retrieval?

| Need | LangGraph benefit |
|------|-------------------|
| **When** to retrieve | Router node or tool-calling agent |
| **Re-retrieve** on bad docs | Conditional edge → rewrite query |
| **Trace** retrieval steps | `graph_path` in state |
| **Checkpoint** conversations | `MemorySaver` + `thread_id` |
| **Human review** before answer | Interrupt after retrieve node |

FAISS stays a **retriever service**. LangGraph decides **orchestration** around it.

---

## 2. Build the FAISS Retriever

Chunk policies → TF-IDF embed → `IndexFlatIP` → `FaissRetriever.search()`.

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    title: str
    text: str


@dataclass
class SearchHit:
    chunk_id: str
    doc_id: str
    title: str
    text: str
    score: float


def chunk_documents(docs: list[dict[str, str]]) -> list[Chunk]:
    return [
        Chunk(chunk_id=f"{d['id']}#0", doc_id=d["id"], title=d["title"], text=d["text"])
        for d in docs
    ]


CHUNKS = chunk_documents(POLICY_DOCS)


def build_tfidf_embeddings(texts: list[str]) -> tuple[TfidfVectorizer, np.ndarray]:
    vectorizer = TfidfVectorizer(stop_words="english")
    dense = vectorizer.fit_transform(texts).astype("float32").toarray()
    faiss.normalize_L2(dense)
    return vectorizer, dense


corpus_texts = [f"{c.title}. {c.text}" for c in CHUNKS]
VECTORIZER, CORPUS_VECTORS = build_tfidf_embeddings(corpus_texts)

FAISS_INDEX = faiss.IndexFlatIP(CORPUS_VECTORS.shape[1])
FAISS_INDEX.add(CORPUS_VECTORS)


class FaissRetriever:
    def __init__(self, index: faiss.Index, vectorizer: TfidfVectorizer, chunks: list[Chunk], min_score: float = 0.05) -> None:
        self.index = index
        self.vectorizer = vectorizer
        self.chunks = chunks
        self.min_score = min_score

    def search(self, query: str, top_k: int = 3) -> list[SearchHit]:
        vec = self.vectorizer.transform([query]).astype("float32").toarray()
        faiss.normalize_L2(vec)
        scores, indices = self.index.search(vec, top_k)
        hits: list[SearchHit] = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or score < self.min_score:
                continue
            c = self.chunks[int(idx)]
            hits.append(SearchHit(c.chunk_id, c.doc_id, c.title, c.text, float(score)))
        return hits

    def to_documents(self, hits: list[SearchHit]) -> list[Document]:
        return [
            Document(page_content=h.text, metadata={"doc_id": h.doc_id, "title": h.title, "score": h.score})
            for h in hits
        ]


RETRIEVER = FaissRetriever(FAISS_INDEX, VECTORIZER, CHUNKS)
demo = RETRIEVER.search("money back guarantee")
print(f"FAISS index: {FAISS_INDEX.ntotal} vectors | demo hit: {demo[0].title if demo else 'none'}")

---

## 3. LangGraph RAG State

Carry both **messages** (for agent/tool loops) and **retrieval fields** (for pipeline graphs).

In [ ]:
class RAGGraphState(TypedDict):
    messages: Annotated[list, add_messages]
    question: str
    retrieval_hits: list[dict[str, Any]]
    context: str
    grade: str
    rewrite_count: int
    graph_path: list[str]
    answer: str


def hits_to_dicts(hits: list[SearchHit]) -> list[dict[str, Any]]:
    return [
        {"chunk_id": h.chunk_id, "doc_id": h.doc_id, "title": h.title, "text": h.text, "score": h.score}
        for h in hits
    ]


def format_context(hits: list[SearchHit]) -> str:
    if not hits:
        return ""
    blocks = [f"[{h.doc_id}] {h.title}: {h.text}" for h in hits]
    return "\n\n".join(blocks)


def format_cited_answer(hits: list[SearchHit], body: str) -> str:
    cites = ", ".join(sorted({h.doc_id for h in hits}))
    return f"{body}\n\nSources: {cites}" if cites else body

---

## 4. Pattern A — Fixed Retrieve → Generate Graph

Always retrieve before answering. Simplest LangGraph RAG — good for narrow domains with high recall needs.

In [ ]:
def retrieve_node(state: RAGGraphState) -> dict[str, Any]:
    hits = RETRIEVER.search(state["question"], top_k=3)
    return {
        "retrieval_hits": hits_to_dicts(hits),
        "context": format_context(hits),
        "graph_path": state.get("graph_path", []) + ["retrieve"],
    }


def generate_node(state: RAGGraphState) -> dict[str, Any]:
    hit_objs = [
        SearchHit(h["chunk_id"], h["doc_id"], h["title"], h["text"], h["score"])
        for h in state["retrieval_hits"]
    ]
    if not hit_objs:
        answer = "I don't have verified policy information for that question."
    else:
        top = hit_objs[0]
        answer = format_cited_answer(hit_objs, f"Based on ShopCo policy: {top.text}")
    return {
        "answer": answer,
        "messages": [AIMessage(content=answer)],
        "graph_path": state["graph_path"] + ["generate"],
    }


pipeline_builder = StateGraph(RAGGraphState)
pipeline_builder.add_node("retrieve", retrieve_node)
pipeline_builder.add_node("generate", generate_node)
pipeline_builder.add_edge(START, "retrieve")
pipeline_builder.add_edge("retrieve", "generate")
pipeline_builder.add_edge("generate", END)

retrieve_generate_graph = pipeline_builder.compile()

out_a = retrieve_generate_graph.invoke({
    "messages": [], "question": "How long until I get my money back?",
    "retrieval_hits": [], "context": "", "grade": "", "rewrite_count": 0,
    "graph_path": [], "answer": "",
})
print("Path:", " → ".join(out_a["graph_path"]))
print(out_a["answer"][:150], "...")

---

## 5. Expose FAISS as a LangChain Tool

Pattern B uses `ToolNode` — the model (or offline planner) decides **when** to search.

In [ ]:
@tool
def search_policy_docs(query: str, top_k: int = 3) -> str:
    """Search ShopCo policy and FAQ documents. Use for returns, shipping, billing, warranty questions."""
    hits = RETRIEVER.search(query, top_k=top_k)
    if not hits:
        return json.dumps({"hits": [], "message": "no relevant documents"})
    return json.dumps({
        "hits": hits_to_dicts(hits),
        "message": f"found {len(hits)} documents",
    })


@tool
def lookup_order(order_id: str) -> str:
    """Look up order status from ShopCo CRM. Use when customer mentions ORD-XXXX."""
    order = ORDERS_DB.get(order_id.upper())
    if not order:
        return json.dumps({"error": f"order {order_id} not found"})
    return json.dumps(order)


RAG_TOOLS = [search_policy_docs, lookup_order]
TOOL_NODE = ToolNode(RAG_TOOLS)
print("Tools:", [t.name for t in RAG_TOOLS])

---

## 6. Pattern B — Agent Graph with FAISS ToolNode

```
START → agent → tools_condition → tools(FAISS) → agent → END
```

In [ ]:
def offline_agent_node(state: RAGGraphState) -> dict[str, Any]:
    has_tools = any(isinstance(m, ToolMessage) for m in state["messages"])
    if has_tools:
        snippets = [str(m.content)[:100] for m in state["messages"] if isinstance(m, ToolMessage)]
        answer = "Here is verified ShopCo information: " + " | ".join(snippets)
        return {"messages": [AIMessage(content=answer)], "answer": answer, "graph_path": state.get("graph_path", []) + ["agent:respond"]}

    q = state["question"].lower()
    tool_calls: list[dict[str, Any]] = []
    if any(t in q for t in ("return", "refund", "shipping", "billing", "warranty", "policy", "money back")):
        tool_calls.append({"name": "search_policy_docs", "args": {"query": state["question"], "top_k": 2}, "id": "tc1", "type": "tool_call"})
    m = re.search(r"ORD-[0-9]{4}", state["question"].upper())
    if m or "order" in q:
        oid = m.group(0) if m else "ORD-1001"
        tool_calls.append({"name": "lookup_order", "args": {"order_id": oid}, "id": "tc2", "type": "tool_call"})
    if not tool_calls:
        tool_calls.append({"name": "search_policy_docs", "args": {"query": state["question"]}, "id": "tc0", "type": "tool_call"})
    return {
        "messages": [AIMessage(content="", tool_calls=tool_calls)],
        "graph_path": state.get("graph_path", []) + ["agent:plan"],
    }


agent_builder = StateGraph(RAGGraphState)
agent_builder.add_node("agent", offline_agent_node)
agent_builder.add_node("tools", TOOL_NODE)
agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges("agent", tools_condition)
agent_builder.add_edge("tools", "agent")

tool_agent_graph = agent_builder.compile()

out_b = tool_agent_graph.invoke({
    "messages": [HumanMessage(content="Return policy and order ORD-1001 status")],
    "question": "Return policy and order ORD-1001 status",
    "retrieval_hits": [], "context": "", "grade": "", "rewrite_count": 0,
    "graph_path": [], "answer": "",
})
print("Final:", out_b["messages"][-1].content[:120], "...")

---

## 7. Document Grading — Corrective RAG Gate

After retrieval, **grade** whether chunks answer the question. If not, rewrite the query and retrieve again.

In [ ]:
def grade_documents(state: RAGGraphState) -> dict[str, Any]:
    """Offline grader: check keyword overlap between question and top chunk."""
    hits = state["retrieval_hits"]
    if not hits:
        return {"grade": "irrelevant", "graph_path": state["graph_path"] + ["grade:none"]}
    q_words = set(re.findall(r"[a-z]{4,}", state["question"].lower()))
    top_text = hits[0]["text"].lower()
    overlap = sum(1 for w in q_words if w in top_text)
    # Semantic queries like "money back" may not overlap — use score as tiebreaker
    score = hits[0]["score"]
    if overlap >= 1 or score >= 0.15:
        grade = "relevant"
    else:
        grade = "irrelevant"
    return {"grade": grade, "graph_path": state["graph_path"] + [f"grade:{grade}"]}


def rewrite_query(state: RAGGraphState) -> dict[str, Any]:
    """Expand vague query for second retrieval pass."""
    q = state["question"]
    expanded = f"{q} ShopCo policy returns shipping billing warranty"
    return {
        "question": expanded,
        "rewrite_count": state["rewrite_count"] + 1,
        "graph_path": state["graph_path"] + ["rewrite"],
    }


def route_after_grade(state: RAGGraphState) -> Literal["generate", "rewrite", "end"]:
    if state["grade"] == "relevant":
        return "generate"
    if state["rewrite_count"] < 1:
        return "rewrite"
    return "generate"  # give up after one rewrite, answer with best effort

---

## 8. Pattern C — Corrective RAG Graph

```
START → retrieve → grade → generate → END
                    └→ rewrite → retrieve (loop once)
```

In [ ]:
corrective_builder = StateGraph(RAGGraphState)
corrective_builder.add_node("retrieve", retrieve_node)
corrective_builder.add_node("grade", grade_documents)
corrective_builder.add_node("rewrite", rewrite_query)
corrective_builder.add_node("generate", generate_node)

corrective_builder.add_edge(START, "retrieve")
corrective_builder.add_edge("retrieve", "grade")
corrective_builder.add_conditional_edges("grade", route_after_grade, {
    "generate": "generate", "rewrite": "rewrite", "end": END,
})
corrective_builder.add_edge("rewrite", "retrieve")
corrective_builder.add_edge("generate", END)

corrective_graph = corrective_builder.compile(checkpointer=MemorySaver())

out_c = corrective_graph.invoke({
    "messages": [], "question": "gadget protection plan",
    "retrieval_hits": [], "context": "", "grade": "", "rewrite_count": 0,
    "graph_path": [], "answer": "",
}, config={"configurable": {"thread_id": "rag-demo-1"}})
print("Path:", " → ".join(out_c["graph_path"]))
print("Grade:", out_c.get("grade"), "| Rewrites:", out_c["rewrite_count"])
print(out_c["answer"][:120], "...")

---

## 9. Router — When to Enter RAG Subgraph

Not every message needs FAISS. Greetings skip retrieval.

In [ ]:
class RouterState(TypedDict):
    question: str
    route: str
    answer: str


def classify_route(state: RouterState) -> dict[str, Any]:
    q = state["question"].lower().strip()
    if q in ("hi", "hello", "thanks", "thank you"):
        return {"route": "chat"}
    if any(t in q for t in ("return", "refund", "shipping", "billing", "warranty", "policy", "order", "ord-")):
        return {"route": "rag"}
    return {"route": "rag"}


def chat_node(state: RouterState) -> dict[str, Any]:
    return {"answer": "Hello! Ask me about ShopCo orders or policies."}


def rag_subgraph_node(state: RouterState) -> dict[str, Any]:
    result = retrieve_generate_graph.invoke({
        "messages": [], "question": state["question"],
        "retrieval_hits": [], "context": "", "grade": "", "rewrite_count": 0,
        "graph_path": [], "answer": "",
    })
    return {"answer": result["answer"]}


def pick_route(state: RouterState) -> str:
    return state["route"]


router_builder = StateGraph(RouterState)
router_builder.add_node("classify", classify_route)
router_builder.add_node("chat", chat_node)
router_builder.add_node("rag", rag_subgraph_node)
router_builder.add_edge(START, "classify")
router_builder.add_conditional_edges("classify", pick_route, {"chat": "chat", "rag": "rag"})
router_builder.add_edge("chat", END)
router_builder.add_edge("rag", END)

router_graph = router_builder.compile()

for q in ["Hello!", "What is the return window?"]:
    r = router_graph.invoke({"question": q, "route": "", "answer": ""})
    print(f"Q: {q} → {r['answer'][:70]}...")

---

## 10. Wiring FAISS Index at Graph Startup

In production, load the serialized index once when the graph worker starts — not per request.

In [ ]:
@dataclass
class RetrieverBundle:
    retriever: FaissRetriever
    version: str = "tfidf-v1"


def load_retriever_bundle() -> RetrieverBundle:
    """Factory called once at worker boot."""
    chunks = chunk_documents(POLICY_DOCS)
    texts = [f"{c.title}. {c.text}" for c in chunks]
    vectorizer, vectors = build_tfidf_embeddings(texts)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return RetrieverBundle(FaissRetriever(index, vectorizer, chunks))


BUNDLE = load_retriever_bundle()
print(f"Loaded retriever bundle {BUNDLE.version} — {BUNDLE.retriever.index.ntotal} vectors")

---

## 11. Abstention on Low Scores

Strict `min_score` prevents citing irrelevant FAISS hits.

In [ ]:
STRICT = FaissRetriever(FAISS_INDEX, VECTORIZER, CHUNKS, min_score=0.35)
vague_hits = STRICT.search("quantum physics refund")
print("Strict retriever hits:", len(vague_hits))
if not vague_hits:
    print("→ Graph should abstain, not hallucinate policy text.")

---

## 12. Run Scenario Suite — All Three Patterns

In [ ]:
SCENARIOS = [
    ("Pipeline", "How long until refund after return?", retrieve_generate_graph),
    ("Tool agent", "ORD-1002 status and warranty info", tool_agent_graph),
    ("Corrective", "electronics coverage plan", corrective_graph),
]

for label, question, graph in SCENARIOS:
    kwargs: dict[str, Any] = {
        "messages": [HumanMessage(content=question)] if label == "Tool agent" else [],
        "question": question,
        "retrieval_hits": [], "context": "", "grade": "", "rewrite_count": 0,
        "graph_path": [], "answer": "",
    }
    config = {"configurable": {"thread_id": f"demo-{label}"}} if label == "Corrective" else None
    result = graph.invoke(kwargs, config=config) if config else graph.invoke(kwargs)
    ans = result.get("answer") or (result["messages"][-1].content if result.get("messages") else "")
    path = result.get("graph_path", [])
    print(f"\n=== {label} ===")
    print(f"Q: {question}")
    print(f"Path: {' → '.join(path) if path else 'agent loop'}")
    print(f"A: {str(ans)[:100]}...")

---

## 13. Pattern Comparison

| Pattern | Graph shape | When to use |
|---------|-------------|-------------|
| **A: retrieve → generate** | Linear 2-node | Always need context; simple Q&A |
| **B: ToolNode** | Agent ⇄ tools | Mixed intents (policy + order lookup) |
| **C: Corrective** | retrieve → grade → rewrite loop | Noisy queries, quality bar |
| **Router + subgraph** | classify → rag or chat | Reduce unnecessary FAISS calls |

---

## 14. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Retrieve inside every prompt string | No trace, no re-query | Dedicated retrieve node or tool |
| Rebuild FAISS per request | High latency | Load bundle at worker startup |
| No score threshold | Wrong citations | `min_score` + grader node |
| Skip router | FAISS on "thanks" | Classify before subgraph |
| Giant context in state | Checkpoint bloat | Store hit ids, rebuild context in generate |

---

## 15. Production Checklist

- [ ] FAISS index loaded once per worker process
- [ ] Retriever exposed as tool **or** explicit graph node (not both hidden in prompts)
- [ ] Citations include `doc_id` from chunk metadata
- [ ] Grader / score gate before generation
- [ ] Router skips retrieval for non-policy intents
- [ ] `MemorySaver` or external checkpointer for multi-turn threads
- [ ] Retrieval logged (query, hits, scores) per graph step

---

## 16. Optional Live LLM Generate Node

Replace template answers in `generate_node` with `ChatOpenAI` when `USE_LIVE_LLM = True`.

In [ ]:
def live_generate_node(state: RAGGraphState) -> dict[str, Any]:
    if not USE_LIVE_LLM:
        return generate_node(state)
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    ctx = state.get("context") or "No documents retrieved."
    prompt = [
        SystemMessage(content=f"Answer using ONLY this ShopCo context. Cite doc ids.\n\n{ctx}"),
        HumanMessage(content=state["question"]),
    ]
    answer = llm.invoke(prompt).content
    return {"answer": answer, "messages": [AIMessage(content=answer)], "graph_path": state["graph_path"] + ["generate:live"]}


print("Live generate ready — set USE_LIVE_LLM=True to replace template answers.")
sample = generate_node({
    "messages": [], "question": "refund timing", "retrieval_hits": hits_to_dicts(RETRIEVER.search("refund timing")),
    "context": format_context(RETRIEVER.search("refund timing")), "grade": "relevant",
    "rewrite_count": 0, "graph_path": ["retrieve"], "answer": "",
})
print(sample["answer"][:120], "...")

---

## 17. Quiz

1. When should FAISS retrieval be a **graph node** vs a **ToolNode**?
2. What does a **grade** node add in corrective RAG?
3. Why load the FAISS index at worker startup?
4. How does LangGraph state carry retrieval results alongside messages?
5. When should a router skip the RAG subgraph entirely?

---

## 18. Summary

**FAISS** is the retrieval engine; **LangGraph** is the control plane. Integrate them by:

1. **Pattern A** — `retrieve` → `generate` for fixed RAG pipelines
2. **Pattern B** — `search_policy_docs` in `ToolNode` for agent-driven retrieval
3. **Pattern C** — `grade` + `rewrite` loop for corrective RAG
4. **Router** — enter RAG only when the intent needs policy grounding

ShopCo Policy Support demonstrates all four with offline planners, citations, abstention, and optional live LLM generation — the same shapes you deploy with serialized FAISS indexes and LangGraph checkpointers in production.